# EDA — news classification

Exploratory Data Analysis собранного датасета из `data/raw/`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Загрузка последнего собранного файла
raw_files = sorted(Path('../data/raw').glob('*.parquet'))
if not raw_files:
    raise FileNotFoundError('Сначала запустите DataCollectionAgent (Шаг 1)')

df = pd.read_parquet(raw_files[-1])
print(f'Загружено: {raw_files[-1]}')
print(f'Размер: {df.shape}')

In [ ]:
# Базовая статистика
print('=== Типы данных ===')
print(df.dtypes)
print('\n=== Пропуски ===')
print(df.isna().sum())
print('\n=== Распределение меток ===')
if 'label' in df.columns:
    print(df['label'].value_counts())

In [ ]:
# Гистограмма длин текстов
if 'text' in df.columns:
    df['text_len'] = df['text'].astype(str).str.len()
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].hist(df['text_len'].clip(0, 2000), bins=50, color='steelblue', edgecolor='white')
    axes[0].set_title('Распределение длин текстов')
    axes[0].set_xlabel('Длина (символов)')
    axes[0].set_ylabel('Количество')
    
    if 'label' in df.columns:
        df['label'].value_counts().plot(kind='bar', ax=axes[1], color='steelblue', edgecolor='white')
        axes[1].set_title('Распределение классов')
        axes[1].set_xlabel('Класс')
        axes[1].set_ylabel('Количество')
        axes[1].tick_params(axis='x', rotation=30)
    
    plt.tight_layout()
    plt.savefig('../reports/eda_distribution.png', dpi=120, bbox_inches='tight')
    plt.show()

In [ ]:
# Примеры из каждого источника
if 'source' in df.columns:
    print('=== Источники ===')
    print(df['source'].value_counts())
    print()
    for src in df['source'].unique():
        print(f'--- Источник: {src} ---')
        sample = df[df['source'] == src][['text', 'label']].head(3)
        for _, row in sample.iterrows():
            print(f'  [{row.get("label", "?")}] {str(row["text"])[:120]}...')
        print()

## Вывод

Датасет собран и готов для передачи в **DataQualityAgent** (Шаг 2).

- Итоговый размер: см. `df.shape` выше
- Источники: ag_news (HuggingFace)
- Классы: World, Sports, Business, Sci/Tech
- Следующий шаг: `python run_pipeline.py` или `/data-quality`